In [1]:
# Клітинка 1. Завантаження CSV-файлу

import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

df = None

path_input = widgets.Text(
    value="",
    placeholder=r"C:\Users\User\Documents\data.csv",
    description="Шлях до CSV:",
    layout=widgets.Layout(width="75%"),
    style={"description_width": "initial"}
)

load_button = widgets.Button(
    description="Завантажити",
    button_style="success",
    icon="upload"
)

load_output = widgets.Output()

def load_csv(_):
    global df
    with load_output:
        clear_output()
        path = path_input.value.strip().strip('"')

        if not path:
            print("Помилка: введіть шлях до CSV-файлу.")
            return
        if not os.path.exists(path):
            print("Помилка: файл не знайдено. Перевірте шлях.")
            return
        if not os.path.isfile(path):
            print("Помилка: вказаний шлях не є файлом.")
            return
        if not path.lower().endswith(".csv"):
            print("Помилка: вибраний файл повинен мати розширення .csv.")
            return

        try:
            loaded = pd.read_csv(path)
        except (pd.errors.ParserError, UnicodeDecodeError, OSError, ValueError) as error:
            print(f"Не вдалося прочитати файл: {error}")
            return
        except Exception as error:
            print(f"Невідома помилка під час читання файлу: {error}")
            return

        if loaded.empty:
            print("Помилка: CSV-файл не містить рядків із даними.")
            return
        if len(loaded.columns) == 0:
            print("Помилка: у файлі відсутні стовпці.")
            return

        numeric_columns = loaded.select_dtypes(include="number").columns.tolist()
        if not numeric_columns:
            print("Помилка: у файлі немає числових стовпців для аналізу.")
            return

        df = loaded
        print(f"Файл успішно завантажено: {os.path.basename(path)}")
        print(f"Розмір таблиці: {df.shape[0]} рядків × {df.shape[1]} стовпців")
        print("\nПерші 5 рядків:")
        display(df.head())
        print("\nЧислові стовпці для аналізу:", ", ".join(map(str, numeric_columns)))

load_button.on_click(load_csv)
display(widgets.VBox([path_input, load_button, load_output]))


In [3]:
# Клітинка 2. Інтерактивний графік і статистика

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# Ця клітинка використовує DataFrame df, який створюється після завантаження CSV у клітинці 1.

if df is None:
    print("Спочатку завантажте CSV-файл у першій клітинці.")
else:
    numeric_columns = df.select_dtypes(include="number").columns.tolist()

    if not numeric_columns:
        print("У DataFrame немає числових стовпців для побудови графіка.")
    else:
        column_widget = widgets.Dropdown(
            options=numeric_columns,
            value=numeric_columns[0],
            description="Стовпець:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="350px")
        )

        line_width_widget = widgets.IntSlider(
            value=2, min=1, max=5, step=1,
            description="Товщина лінії:",
            continuous_update=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="350px")
        )

        start_widget = widgets.IntSlider(
            value=0, min=0, max=max(len(df) - 1, 0), step=1,
            description="Початковий індекс:",
            continuous_update=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="450px")
        )

        end_widget = widgets.IntSlider(
            value=max(len(df) - 1, 0), min=0, max=max(len(df) - 1, 0), step=1,
            description="Кінцевий індекс:",
            continuous_update=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="450px")
        )

        stats_output = widgets.Output()
        plot_output = widgets.Output()

        def update_analysis(change=None):
            start = start_widget.value
            end = end_widget.value

            # Гарантуємо правильний порядок індексів, навіть якщо користувач поміняв їх місцями.
            left, right = min(start, end), max(start, end)
            values = pd.to_numeric(df[column_widget.value], errors="coerce").iloc[left:right + 1]
            values = values.dropna()

            with stats_output:
                clear_output(wait=True)
                if values.empty:
                    print("У вибраному діапазоні немає числових значень для аналізу.")
                else:
                    stats = [
                        ("Кількість значень", f"{values.count()}"),
                        ("Мінімум", f"{values.min():.4g}"),
                        ("Максимум", f"{values.max():.4g}"),
                        ("Середнє", f"{values.mean():.4g}"),
                        ("Медіана", f"{values.median():.4g}")
                    ]
                    display(widgets.VBox([
                        widgets.Label(f"{name}: {value}") for name, value in stats
                    ]))

            with plot_output:
                clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(10, 4.5))
                plot_values = pd.to_numeric(
                    df[column_widget.value].iloc[left:right + 1], errors="coerce"
                )
                x = np.arange(left, right + 1)
                valid = plot_values.notna()
                if valid.any():
                    ax.plot(
                        x[valid], plot_values[valid],
                        linewidth=line_width_widget.value,
                        marker="o",
                        markersize=3
                    )
                    ax.set_title(f"Графік стовпця «{column_widget.value}»")
                    ax.set_xlabel("Індекс рядка")
                    ax.set_ylabel(str(column_widget.value))
                    ax.grid(True, alpha=0.3)
                else:
                    ax.text(0.5, 0.5, "Немає числових значень у вибраному діапазоні",
                            ha="center", va="center", transform=ax.transAxes)
                fig.tight_layout()
                plt.show()
                plt.close(fig)

        for control in (column_widget, line_width_widget, start_widget, end_widget):
            control.observe(update_analysis, names="value")

        controls = widgets.VBox([
            widgets.HBox([column_widget, line_width_widget]),
            start_widget,
            end_widget
        ])
        display(controls)
        display(widgets.HTML("<h4>Статистичні характеристики вибраного діапазону</h4>"))
        display(stats_output)
        display(plot_output)
        update_analysis()


HTML(value='<h4>Статистичні характеристики вибраного діапазону</h4>')

Output()

Output()